In [4]:
import numpy as np
import pandas as pd
import joblib

X_test_clean = pd.read_csv("../data/X_test_clean.csv")
y_test = pd.read_csv("../data/y_test.csv").values.ravel()
xgb = joblib.load("../models/final_model_xgb_clean.pkl")
threshold = joblib.load("../models/ldecision_threshold.pkl")
xgb_proba = xgb.predict_proba(X_test_clean)[:, 1]

In [8]:
X_test_clean['credit_score'].describe()

count    29734.000000
mean        -0.001119
std          1.003152
min         -1.725477
25%         -0.870575
50%         -0.007038
75%          0.882405
max          1.728672
Name: credit_score, dtype: float64

In [9]:
df_err = X_test_clean.copy()
df_err['actual'] = y_test
df_err['pred'] = (xgb_proba >= 0.41).astype(int)

# z-score bands: split at the 33rd/66th percentile-ish points
df_err['score_band'] = pd.cut(df_err['credit_score'],
                              bins=[-5, -0.6, 0.6, 5],
                              labels=['low', 'medium', 'high'])

for band in ['low', 'medium', 'high']:
    sub = df_err[(df_err['score_band'] == band) & (df_err['actual'] == 1)]
    if len(sub) > 0:
        recall = sub['pred'].sum() / len(sub)
        print(f"{band:>6}: recall = {recall:.2f}  (n={len(sub)} defaulters)")

   low: recall = 0.77  (n=2420 defaulters)
medium: recall = 0.76  (n=2454 defaulters)
  high: recall = 0.79  (n=2454 defaulters)


In [10]:
def score_applicant(prob):
    """Convert default probability -> 0-100 risk score + decision band."""
    risk_score = round(prob * 100)
    if prob < 0.30:
        band = "Approve"
    elif prob < 0.55:
        band = "Manual Review"
    else:
        band = "Reject"
    return risk_score, band

# test on a few real applicants from the test set
sample_idx = [0, 5, 10, 50, 100]
for i in sample_idx:
    prob = xgb_proba[i]
    score, band = score_applicant(prob)
    actual = "DEFAULTED" if y_test[i] == 1 else "repaid"
    print(f"Applicant {i}: risk={score:>3}/100 -> {band:<14} (actual: {actual})")

Applicant 0: risk=  6/100 -> Approve        (actual: repaid)
Applicant 5: risk= 84/100 -> Reject         (actual: repaid)
Applicant 10: risk= 18/100 -> Approve        (actual: repaid)
Applicant 50: risk= 60/100 -> Reject         (actual: repaid)
Applicant 100: risk=100/100 -> Reject         (actual: DEFAULTED)
